# Milestone 5 — XGBoost comparison

Start with a bounded configuration: 300 trees, depth 3, learning rate 0.05, and CPU histogram training. Evaluate the same untouched validation partition as the baselines. Test performance remains unavailable until selection is complete.

In [1]:
from pathlib import Path
import nbformat
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/raw/ai4i2020.csv").exists())
notebook = nbformat.read(PROJECT_ROOT / "notebooks/04_baseline_models.ipynb", as_version=4)
for cell in notebook.cells:
    if cell.cell_type == "code" and "definitions" in cell.metadata.get("tags", []):
        exec(compile(cell.source, "baseline_definitions", "exec"), globals())
from xgboost import XGBClassifier
records, manifest = load_partitions()
X_train, y_train = get_partition(records, manifest, "train")
X_validation, y_validation = get_partition(records, manifest, "validation")

In [2]:
estimator = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.9, colsample_bytree=1.0, objective="binary:logistic", eval_metric="logloss", tree_method="hist", n_jobs=4, random_state=RANDOM_STATE)
pipeline = train_pipeline(estimator, X_train, y_train)
probabilities = pipeline.predict_proba(X_validation)[:, 1]
result = {"model": "xgboost", "partition": "validation", **evaluate_probabilities(y_validation, probabilities)}
joblib.dump(pipeline, MODEL_DIR / "xgboost.joblib")
(METRICS_DIR / "xgboost.json").write_text(json.dumps(result, indent=2) + "\n")
comparison = json.loads((METRICS_DIR / "baselines.json").read_text()) + [result]
(METRICS_DIR / "model_comparison.json").write_text(json.dumps(comparison, indent=2) + "\n")
display(pd.DataFrame(comparison).drop(columns="confusion_matrix"))
assert np.isfinite(probabilities).all()
np.testing.assert_allclose(joblib.load(MODEL_DIR / "xgboost.joblib").predict_proba(X_validation), pipeline.predict_proba(X_validation))

,model,partition,threshold,precision,recall,f1,f2,roc_auc,pr_auc,brier_score,false_negatives,false_negative_rate,support,positive_support
0,logistic_regression,validation,0.5,0.727273,0.235294,0.355556,0.272109,0.873714,0.432080,0.024883,52,0.764706,2000,68
1,random_forest,validation,0.5,0.852941,0.426471,0.568627,0.473856,0.955795,0.745368,0.016188,39,0.573529,2000,68
2,xgboost,validation,0.5,0.893617,0.617647,0.730435,0.658307,0.965481,0.794102,0.013479,26,0.382353,2000,68
